In [5]:
from google import genai
from google.genai import types
import wave
from loguru import logger
from pathlib import Path
import json

# load the data 

In [9]:
BASE_DIR = Path.cwd().parent.resolve()

In [10]:
BASE_DIR

WindowsPath('D:/GAN_AI/Synthetic-Speech-Data-Pipeline-For-STT')

In [29]:
DATA_PATH = Path(r"..\data\accepted.jsonl")

NOISE_PATH = Path(r"..\data\background_noise")
DATA_PATH

WindowsPath('../data/accepted.jsonl')

In [30]:
NOISE_PATH/"sd.wav"

WindowsPath('../data/background_noise/sd.wav')

In [31]:

audio_data = []

with open(DATA_PATH, "r" , encoding="utf-8") as f:
    for line in f:
        audio_data.append(json.loads(line))

print(audio_data)

[{'audio_id': '759c7a5b-78a7-4749-aeb1-6122cb88b9ec', 'prompt_id': '759c7a5b-78a7-4749-aeb1-6122cb88b9ec', 'audio_path': 'data\\audio_outputs\\759c7a5b-78a7-4749-aeb1-6122cb88b9ec.wav', 'text': 'لو سمحت، هو البنطلون ده بكام؟', 'voice_name': 'Algenib', 'speaker_id': '3', 'tts_model': 'gemini-2.5-flash-preview-tts', 'background_noise': 'background street noise', 'sample_rate': 24000, 'status': 'reviewed', 'review_status': 'accepted', 'duration_seconds': 2.9309583333333333, 'wer': 0.0, 'transcript': 'لو سمحت هو البنطلون ده بكام', 'issues': [], 'validated_at': '2026-05-11T18:59:43.306899'}, {'audio_id': 'ef2b9043-9163-4aa8-8289-8ff1e22ce3f5', 'prompt_id': 'ef2b9043-9163-4aa8-8289-8ff1e22ce3f5', 'audio_path': 'data\\audio_outputs\\ef2b9043-9163-4aa8-8289-8ff1e22ce3f5.wav', 'text': 'لو سمحت، عايز طبق كشري كبير ومش عايزه سبايسي خالص، وشكرا.', 'voice_name': 'Algieba', 'speaker_id': '4', 'tts_model': 'gemini-2.5-flash-preview-tts', 'background_noise': 'background crowd', 'sample_rate': 24000, '

In [32]:
Path(audio_data[1]['audio_path'])

WindowsPath('data/audio_outputs/ef2b9043-9163-4aa8-8289-8ff1e22ce3f5.wav')

In [33]:
import numpy as np
import soundfile as sf
import random
from scipy.signal import resample_poly


def add_background_noise(
    speech_path: str,
    noise_path: str,
    output_path: str,
    snr_db: float = 10,
):
    # Load
    speech, sr = sf.read(speech_path)
    noise, noise_sr = sf.read(noise_path)

    # Mono
    if speech.ndim > 1:
        speech = speech.mean(axis=1)

    if noise.ndim > 1:
        noise = noise.mean(axis=1)

    speech = speech.astype(np.float32)
    noise = noise.astype(np.float32)

    # 🚀 FAST resampling (key speed improvement)
    if noise_sr != sr:
        gcd = np.gcd(noise_sr, sr)
        noise = resample_poly(noise, sr // gcd, noise_sr // gcd)

    # Repeat if needed
    if len(noise) < len(speech):
        repeats = (len(speech) // len(noise)) + 1
        noise = np.tile(noise, repeats)

    # Random crop
    start = random.randint(0, len(noise) - len(speech))
    noise = noise[start:start + len(speech)]

    # Power (can be slightly optimized later, but fine)
    speech_power = np.mean(speech * speech)
    noise_power = np.mean(noise * noise)

    scale = np.sqrt(
        speech_power / (10 ** (snr_db / 10) * noise_power + 1e-9)
    )

    noise *= scale

    mixed = speech + noise

    # Normalize safely
    peak = np.max(np.abs(mixed)) + 1e-9
    mixed = mixed / peak

    sf.write(output_path, mixed, sr)

    return output_path

In [34]:
NOISE_PATH / "crowd_noise.wav"

WindowsPath('../data/background_noise/crowd_noise.wav')

In [35]:
audio_data[1]['background_noise']

'background crowd'

In [36]:
if audio_data[1]['background_noise'] == "background street noise":
    noise_file = "street_noise.wav"
elif audio_data[1]['background_noise'] == "background crowd":
    noise_file = "crowd_noise.wav" 
else:
    pass



add_background_noise(
    speech_path=".." / Path( audio_data[1]['audio_path']),
    noise_path= NOISE_PATH / noise_file ,
    output_path=audio_data[1]['audio_path'].split("\\")[-1].split(".")[0] + "_with_noise.wav",
    snr_db=10,
)

'ef2b9043-9163-4aa8-8289-8ff1e22ce3f5_with_noise.wav'

# function

In [37]:
def save_to_jsonl(output_file_path , record: dict):
        with open(output_file_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

In [45]:
def load_existing_audio_ids(
    metadata_path: Path
) -> set[str]:

    existing_ids = set()

    if not metadata_path.exists():

        return existing_ids

    with open(
        metadata_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            try:

                record = json.loads(line)

                existing_ids.add(
                    record["audio_id"]
                )

            except Exception:

                continue

    return existing_ids

In [38]:
NOISE_MAP = {

    "background street noise": "street_noise.wav",

    "background crowd": "crowd_noise.wav",

}

In [48]:
import uuid
import json
from copy import deepcopy
from pathlib import Path
from datetime import datetime

from loguru import logger


FINAL_METADATA_PATH = Path(
    "../data/final_dataset_metadata.jsonl"
)

EXISTING_AUDIO_IDS = load_existing_audio_ids(
    FINAL_METADATA_PATH
)

In [49]:
EXISTING_AUDIO_IDS

set()

In [50]:
import json
from copy import deepcopy
from pathlib import Path
from datetime import datetime

from loguru import logger


# =========================================================
# CONFIG
# =========================================================

BASE_DIR = Path.cwd().parent

FINAL_METADATA_PATH = Path(
    "../data/final_dataset_metadata.jsonl"
)

NOISE_MAP = {

    "background street noise": "street_noise.wav",

    "background crowd": "crowd_noise.wav",
}


# =========================================================
# LOAD EXISTING IDS
# =========================================================

def load_existing_audio_ids(
    metadata_path: Path
) -> set[str]:

    existing_ids = set()

    if not metadata_path.exists():

        return existing_ids

    with open(
        metadata_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            try:

                record = json.loads(line)

                existing_ids.add(
                    record["audio_id"]
                )

            except Exception:

                continue

    return existing_ids


# global cache
EXISTING_AUDIO_IDS = load_existing_audio_ids(
    FINAL_METADATA_PATH
)


# =========================================================
# SAVE JSONL
# =========================================================

def save_to_jsonl(
    output_path: Path,
    record: dict
):

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        output_path,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )


# =========================================================
# AUGMENT SINGLE RECORD
# =========================================================

def augment_audio_record(

    record: dict,

    noise_dir: Path,

    snr_db: float = 10,
):

    try:

        # =====================================
        # DEEP COPY
        # =====================================

        original_record = deepcopy(record)

        audio_id = original_record["audio_id"]

        # =====================================
        # SKIP IF ALREADY EXISTS
        # =====================================

        if audio_id in EXISTING_AUDIO_IDS:

            logger.warning(
                f"Skipping existing record | "
                f"id={audio_id}"
            )

            return None

        # =====================================
        # CLEAN AUDIO
        # =====================================

        if original_record["background_noise"] == "clean audio":

            logger.info(
                f"Saving clean sample only | "
                f"id={audio_id}"
            )

            original_record["with_noise"] = False

            original_record["created_at"] = (
                datetime.utcnow().isoformat()
            )

            save_to_jsonl(
                FINAL_METADATA_PATH,
                original_record
            )

            EXISTING_AUDIO_IDS.add(audio_id)

            logger.success(
                f"Clean sample saved | "
                f"id={audio_id}"
            )

            return original_record

        # =====================================
        # AUGMENTED ID
        # =====================================

        augmented_audio_id = (
            audio_id + "_with_noise"
        )

        # =====================================
        # SKIP IF AUGMENTED EXISTS
        # =====================================

        if augmented_audio_id in EXISTING_AUDIO_IDS:

            logger.warning(
                f"Skipping existing augmented sample | "
                f"id={augmented_audio_id}"
            )

            return None

        # =====================================
        # GET NOISE FILE
        # =====================================

        noise_type = original_record["background_noise"]

        noise_file = NOISE_MAP.get(noise_type)

        if noise_file is None:

            logger.warning(
                f"Unknown noise type | "
                f"{noise_type}"
            )

            return None

        noise_path = (
            noise_dir / noise_file
        ).resolve()

        if not noise_path.exists():

            logger.error(
                f"Noise file missing | "
                f"{noise_path}"
            )

            return None

        # =====================================
        # ORIGINAL AUDIO PATH
        # =====================================

        original_audio_path = (
            BASE_DIR / original_record["audio_path"]
        ).resolve()

        if not original_audio_path.exists():

            logger.error(
                f"Original audio missing | "
                f"{original_audio_path}"
            )

            return None

        logger.info(
            f"Original audio path | "
            f"{original_audio_path}"
        )

        # =====================================
        # OUTPUT PATH
        # =====================================

        output_path = original_audio_path.with_name(
            original_audio_path.stem
            + "_with_noise.wav"
        )

        # =====================================
        # SKIP IF FILE EXISTS
        # =====================================

        if output_path.exists():

            logger.warning(
                f"Augmented wav already exists | "
                f"{output_path}"
            )

        else:

            # =====================================
            # AUGMENT AUDIO
            # =====================================

            logger.info(
                f"Adding background noise | "
                f"id={audio_id} | "
                f"noise={noise_type}"
            )

            add_background_noise(

                speech_path=str(original_audio_path),

                noise_path=str(noise_path),

                output_path=str(output_path),

                snr_db=snr_db,
            )

            # verify save

            if not output_path.exists():

                raise FileNotFoundError(
                    f"Failed to create augmented file: "
                    f"{output_path}"
                )

            logger.success(
                f"Augmented wav saved | "
                f"{output_path}"
            )

        # =====================================
        # CREATE NEW RECORD
        # =====================================

        augmented_record = deepcopy(
            original_record
        )

        augmented_record["parent_audio_id"] = (
            audio_id
        )

        augmented_record["audio_id"] = (
            augmented_audio_id
        )

        augmented_record["with_noise"] = True

        augmented_record["audio_path"] = str(
            output_path.relative_to(BASE_DIR)
        )

        augmented_record["augmentation"] = {

            "type": "background_noise",

            "noise_type": noise_type,

            "snr_db": snr_db,
        }

        augmented_record["created_at"] = (
            datetime.utcnow().isoformat()
        )

        # =====================================
        # SAVE METADATA
        # =====================================

        save_to_jsonl(
            FINAL_METADATA_PATH,
            augmented_record
        )

        EXISTING_AUDIO_IDS.add(
            augmented_audio_id
        )

        logger.success(
            f"Augmented sample saved | "
            f"id={augmented_audio_id}"
        )

        return augmented_record

    except Exception:

        logger.exception(
            f"Augmentation failed | "
            f"id={record.get('audio_id')}"
        )

        return None


# =========================================================
# AUGMENT DATASET
# =========================================================

def augment_dataset(

    records: list[dict],

    noise_dir: Path,
):

    augmented_records = []

    logger.info(
        f"Starting augmentation | "
        f"samples={len(records)}"
    )

    for record in records:

        result = augment_audio_record(

            record=record,

            noise_dir=noise_dir,
        )

        if result is not None:

            augmented_records.append(result)

    logger.success(
        f"Augmentation completed | "
        f"generated={len(augmented_records)}"
    )

    return augmented_records

In [51]:
audio_data

[{'audio_id': '759c7a5b-78a7-4749-aeb1-6122cb88b9ec',
  'prompt_id': '759c7a5b-78a7-4749-aeb1-6122cb88b9ec',
  'audio_path': 'data\\audio_outputs\\759c7a5b-78a7-4749-aeb1-6122cb88b9ec.wav',
  'text': 'لو سمحت، هو البنطلون ده بكام؟',
  'voice_name': 'Algenib',
  'speaker_id': '3',
  'tts_model': 'gemini-2.5-flash-preview-tts',
  'background_noise': 'background street noise',
  'sample_rate': 24000,
  'status': 'reviewed',
  'review_status': 'accepted',
  'duration_seconds': 2.9309583333333333,
  'wer': 0.0,
  'transcript': 'لو سمحت هو البنطلون ده بكام',
  'issues': [],
  'validated_at': '2026-05-11T18:59:43.306899'},
 {'audio_id': 'ef2b9043-9163-4aa8-8289-8ff1e22ce3f5',
  'prompt_id': 'ef2b9043-9163-4aa8-8289-8ff1e22ce3f5',
  'audio_path': 'data\\audio_outputs\\ef2b9043-9163-4aa8-8289-8ff1e22ce3f5.wav',
  'text': 'لو سمحت، عايز طبق كشري كبير ومش عايزه سبايسي خالص، وشكرا.',
  'voice_name': 'Algieba',
  'speaker_id': '4',
  'tts_model': 'gemini-2.5-flash-preview-tts',
  'background_noise'

In [54]:
audio_data[2]
# augment_audio_record(audio_data[2] , NOISE_PATH )

augment_dataset(audio_data , NOISE_PATH)

2026-05-12 10:03:46.009 | INFO     | __main__:augment_dataset:373 - Starting augmentation | samples=3
2026-05-12 10:03:46.010 | WARNING  | __main__:augment_audio_record:179 - Skipping existing augmented sample | id=759c7a5b-78a7-4749-aeb1-6122cb88b9ec_with_noise
2026-05-12 10:03:46.011 | WARNING  | __main__:augment_audio_record:179 - Skipping existing augmented sample | id=ef2b9043-9163-4aa8-8289-8ff1e22ce3f5_with_noise
2026-05-12 10:03:46.011 | WARNING  | __main__:augment_audio_record:127 - Skipping existing record | id=18c09028-3d0d-4fc1-84cb-2297587f60d8
2026-05-12 10:03:46.014 | SUCCESS  | __main__:augment_dataset:391 - Augmentation completed | generated=0


[]

# class

In [55]:
import json
from copy import deepcopy
from pathlib import Path
from datetime import datetime

from loguru import logger


# =========================================================
# CONFIG
# =========================================================

BASE_DIR = Path.cwd().parent

FINAL_METADATA_PATH = Path(
    "../data/final_dataset_metadata.jsonl"
)

NOISE_MAP = {

    "background street noise": "street_noise.wav",

    "background crowd": "crowd_noise.wav",
}


# =========================================================
# UTILS
# =========================================================

def load_existing_audio_ids(
    metadata_path: Path
) -> set[str]:

    existing_ids = set()

    if not metadata_path.exists():

        return existing_ids

    with open(
        metadata_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            try:

                record = json.loads(line)

                existing_ids.add(
                    record["audio_id"]
                )

            except Exception:

                continue

    return existing_ids


def save_to_jsonl(
    output_path: Path,
    record: dict
):

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        output_path,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )


# =========================================================
# AUGMENTATION SERVICE
# =========================================================

class AudioAugmentationService:

    def __init__(

        self,

        metadata_output_path: Path,

        noise_dir: Path,

        noise_map: dict,

        snr_db: float = 10,
    ):

        self.metadata_output_path = (
            metadata_output_path
        )

        self.noise_dir = noise_dir

        self.noise_map = noise_map

        self.snr_db = snr_db

        # load existing ids once
        self.existing_audio_ids = (
            load_existing_audio_ids(
                metadata_output_path
            )
        )

        logger.info(
            f"AudioAugmentationService initialized | "
            f"existing_records={len(self.existing_audio_ids)}"
        )

    # =====================================================
    # AUGMENT SINGLE RECORD
    # =====================================================

    def augment_audio_record(
        self,
        record: dict,
    ):

        try:

            original_record = deepcopy(record)

            audio_id = original_record["audio_id"]

            # =====================================
            # SKIP EXISTING CLEAN RECORD
            # =====================================

            if audio_id in self.existing_audio_ids:

                logger.warning(
                    f"Skipping existing record | "
                    f"id={audio_id}"
                )

                return None

            # =====================================
            # CLEAN AUDIO
            # =====================================

            if original_record["background_noise"] == "clean audio":

                logger.info(
                    f"Saving clean sample only | "
                    f"id={audio_id}"
                )

                original_record["with_noise"] = False

                original_record["created_at"] = (
                    datetime.utcnow().isoformat()
                )

                save_to_jsonl(
                    self.metadata_output_path,
                    original_record
                )

                self.existing_audio_ids.add(
                    audio_id
                )

                logger.success(
                    f"Clean sample saved | "
                    f"id={audio_id}"
                )

                return original_record

            # =====================================
            # AUGMENTED ID
            # =====================================

            augmented_audio_id = (
                audio_id + "_with_noise"
            )

            # =====================================
            # SKIP EXISTING AUGMENTED RECORD
            # =====================================

            if augmented_audio_id in self.existing_audio_ids:

                logger.warning(
                    f"Skipping existing augmented sample | "
                    f"id={augmented_audio_id}"
                )

                return None

            # =====================================
            # GET NOISE FILE
            # =====================================

            noise_type = (
                original_record["background_noise"]
            )

            noise_file = self.noise_map.get(
                noise_type
            )

            if noise_file is None:

                logger.warning(
                    f"Unknown noise type | "
                    f"{noise_type}"
                )

                return None

            noise_path = (
                self.noise_dir / noise_file
            ).resolve()

            if not noise_path.exists():

                logger.error(
                    f"Noise file missing | "
                    f"{noise_path}"
                )

                return None

            # =====================================
            # ORIGINAL AUDIO
            # =====================================

            original_audio_path = (
                BASE_DIR / original_record["audio_path"]
            ).resolve()

            if not original_audio_path.exists():

                logger.error(
                    f"Original audio missing | "
                    f"{original_audio_path}"
                )

                return None

            # =====================================
            # OUTPUT PATH
            # =====================================

            output_path = (
                original_audio_path.with_name(
                    original_audio_path.stem
                    + "_with_noise.wav"
                )
            )

            # =====================================
            # CREATE WAV
            # =====================================

            if output_path.exists():

                logger.warning(
                    f"Augmented wav already exists | "
                    f"{output_path}"
                )

            else:

                logger.info(
                    f"Adding background noise | "
                    f"id={audio_id} | "
                    f"noise={noise_type}"
                )

                add_background_noise(

                    speech_path=str(
                        original_audio_path
                    ),

                    noise_path=str(
                        noise_path
                    ),

                    output_path=str(
                        output_path
                    ),

                    snr_db=self.snr_db,
                )

                if not output_path.exists():

                    raise FileNotFoundError(
                        f"Failed to create augmented wav | "
                        f"{output_path}"
                    )

                logger.success(
                    f"Augmented wav created | "
                    f"{output_path}"
                )

            # =====================================
            # CREATE METADATA
            # =====================================

            augmented_record = deepcopy(
                original_record
            )

            augmented_record[
                "parent_audio_id"
            ] = audio_id

            augmented_record[
                "audio_id"
            ] = augmented_audio_id

            augmented_record[
                "with_noise"
            ] = True

            augmented_record[
                "audio_path"
            ] = str(
                output_path.relative_to(
                    BASE_DIR
                )
            )

            augmented_record[
                "augmentation"
            ] = {

                "type": "background_noise",

                "noise_type": noise_type,

                "snr_db": self.snr_db,
            }

            augmented_record[
                "created_at"
            ] = datetime.utcnow().isoformat()

            # =====================================
            # SAVE
            # =====================================

            save_to_jsonl(
                self.metadata_output_path,
                augmented_record
            )

            self.existing_audio_ids.add(
                augmented_audio_id
            )

            logger.success(
                f"Augmented sample saved | "
                f"id={augmented_audio_id}"
            )

            return augmented_record

        except Exception:

            logger.exception(
                f"Augmentation failed | "
                f"id={record.get('audio_id')}"
            )

            return None

    # =====================================================
    # AUGMENT DATASET
    # =====================================================

    def augment_dataset(
        self,
        records: list[dict],
    ):

        augmented_records = []

        logger.info(
            f"Starting augmentation | "
            f"samples={len(records)}"
        )

        for record in records:

            result = self.augment_audio_record(
                record
            )

            if result is not None:

                augmented_records.append(
                    result
                )

        logger.success(
            f"Augmentation completed | "
            f"generated={len(augmented_records)}"
        )

        return augmented_records

In [56]:
augmentor = AudioAugmentationService(

    metadata_output_path=FINAL_METADATA_PATH,

    noise_dir=Path("../data/background_noise"),

    noise_map=NOISE_MAP,

    snr_db=10,
)

results = augmentor.augment_dataset(
    audio_data
)

2026-05-12 10:07:23.568 | INFO     | __main__:__init__:124 - AudioAugmentationService initialized | existing_records=3
2026-05-12 10:07:23.572 | INFO     | __main__:augment_dataset:404 - Starting augmentation | samples=3
2026-05-12 10:07:23.573 | WARNING  | __main__:augment_audio_record:204 - Skipping existing augmented sample | id=759c7a5b-78a7-4749-aeb1-6122cb88b9ec_with_noise
2026-05-12 10:07:23.574 | WARNING  | __main__:augment_audio_record:204 - Skipping existing augmented sample | id=ef2b9043-9163-4aa8-8289-8ff1e22ce3f5_with_noise
2026-05-12 10:07:23.574 | WARNING  | __main__:augment_audio_record:150 - Skipping existing record | id=18c09028-3d0d-4fc1-84cb-2297587f60d8
2026-05-12 10:07:23.576 | SUCCESS  | __main__:augment_dataset:421 - Augmentation completed | generated=0
